# Adrianic v4.5 — Multi-Access Memory + Identity Binding State

The memory object now keeps separate consolidation and retrieval access values.

The explicit binding primitive is generic: exact token identity, rarity, observed order, and learned retrieval access. No relation names, chain IDs, answers, or semantic labels are used by the join.

A compact recurrence-gated identity graph is also tested as a structural representation lane; it stores token-identity edges rather than duplicate raw templates.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter, defaultdict
import argparse, copy, json, math, random, re, urllib.request, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# -----------------------------------------------------------------------------
# Adrianic v4.4 — Multi-consequence value + adaptive-gate binding test:
#   (1) pairwise + latent common-cause evidence control (v3.21 family)
#   (2) comparative replacement + honest address/template separation (field gate)
#
# Scientific boundary: this is a tiny recurrent LM + explicit memory controller.
# It tests functional relational binding under interference, not human understanding.
# -----------------------------------------------------------------------------

if Path('/content').exists(): ROOT = Path('/content')
elif Path('/mnt/data').exists(): ROOT = Path('/mnt/data')
else: ROOT = Path.cwd()
OUT = ROOT / 'adrianic_v44_multiconsequence_results'
OUT.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")
STOP = set('the a an is are in of to and this that with for near under before after through beyond fact remember'.split())

def seed_all(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def tokenize(s:str): return TOKEN_RE.findall(s.lower())

def content_tokens(s:str):
    return [w for w in tokenize(s) if w.isalpha() and w not in STOP]

FACTS=[
('two plus two equals','four'),('three plus five equals','eight'),('seven minus three equals','four'),('six times seven equals','fortytwo'),
('the square root of nine is','three'),('a triangle has','three'),('a square has','four'),('the additive identity is','zero'),
('the si unit of force is the','newton'),('the si unit of energy is the','joule'),('the si unit of power is the','watt'),('mass times acceleration gives','force'),
('energy associated with motion is','kinetic'),('the force attracting masses is','gravity'),('the chemical symbol for hydrogen is','h'),
('the chemical symbol for oxygen is','o'),('water contains hydrogen and','oxygen'),('an atom with six protons is','carbon'),
('an atom with eight protons is','oxygen'),('the center of an atom is the','nucleus'),('earth orbits the','sun'),('the moon orbits the','earth'),
('the largest planet in the solar system is','jupiter'),('plants use light in','photosynthesis')]
REPHRASE=[
('in mechanics mass times acceleration produces','force'),('force is measured using the unit called','newton'),
('energy uses the si unit called','joule'),('water contains hydrogen plus','oxygen'),('six protons identifies','carbon'),
('eight protons identifies','oxygen'),('earth travels around the','sun'),('the moon travels around the','earth')]

SYL1=['nar','vel','tor','lum','kas','mer','sol','dav','pir','zan','fel','rin','bos','kel','vim','jor','tav','qul','ser','pan','dru','hel','cor','nis']
SYL2=['a','en','or','is','um','et','on','ar','il','os','ex','un']

def unique_words(n,seed):
    rng=random.Random(seed); pool=[a+b for a in SYL1 for b in SYL2]; rng.shuffle(pool); return pool[:n]

def make_chains(seed=0,n_demo=20,n_probe=10):
    names=unique_words((n_demo+n_probe)*3,seed+818); demo=[]; probe=[]; rows=[]; k=0
    for i in range(n_demo+n_probe):
        x,y,z=names[k:k+3]; k+=3
        premises=[f'{x} contains {y}.',f'{y} maps to {z}.']
        if i<n_demo:
            # Preserve v4.2 curriculum exactly: sentences are independent training items.
            demo += (premises+[f'{x} ultimately maps to {z}.',f'{y} is contained by {x}.'])*3
            rows.append(('demo',x,y,z))
        else:
            # Probe target and inverse sentences are deliberately absent.
            probe += premises*4
            rows.append(('probe',x,y,z))
    return demo,probe,pd.DataFrame(rows,columns=['kind','x','y','z'])

def make_training_corpus(seed=0,contextual_demo=False):
    rng=random.Random(seed); s=[]
    for p,a in FACTS: s += [f'{p} {a}.',f'fact: {p} {a}.',f'remember {p} {a}.']
    demo,probe,chains=make_chains(seed); s += demo+probe
    if contextual_demo:
        # Schema-ingestion patch: demonstration entities additionally appear in a contiguous
        # premise->premise->composition context. Probe entities still NEVER receive the
        # composed target or a combined two-premise paragraph during training.
        for _,r in chains[chains.kind=='demo'].iterrows():
            para=f'{r.x} contains {r.y}. {r.y} maps to {r.z}. {r.x} ultimately maps to {r.z}.'
            s += [para,para]
    for i in range(80): s.append(f'noise{i} drift{i} hollow{i} amber{i} sevenfold{i}.')
    rng.shuffle(s)
    return s,chains

def proxy_stream(seed=0,n=180):
    rng=random.Random(seed+500)
    names=['arden','cassian','lyra','orion','mara','selene','tavian','viola']
    verbs=['whispers','wanders','argues','laughs','waits','returns','dreams','speaks']
    objs=['beside the gate','through the hall','near the river','under the tower','after the feast','beyond the garden','before the crowd','under the moon']
    return [f'{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objs)}.' for _ in range(n)]

def download_natural():
    urls={'shakespeare':'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
          'bible':'https://www.gutenberg.org/cache/epub/10900/pg10900.txt'}
    out={}
    for name,url in urls.items():
        p=ROOT/f'{name}_corpus.txt'
        if not p.exists():
            print('downloading',name); urllib.request.urlretrieve(url,p)
        text=p.read_text(errors='ignore')
        out[name]=[q.strip().replace('\n',' ') for q in re.split(r'(?<=[.!?])\s+|\n{2,}',text) if 4<=len(tokenize(q))<=70]
    return out

def natural_stream(seed=0,corpus='interleaved',per_source=700):
    d=download_natural(); rng=random.Random(seed+900)
    A=rng.sample(d['shakespeare'],min(per_source,len(d['shakespeare'])))
    B=rng.sample(d['bible'],min(per_source,len(d['bible'])))
    if corpus=='shakespeare': return A
    if corpus=='bible': return B
    out=[]; n=min(len(A),len(B))
    for a,b in zip(A[:n],B[:n]): out += ([a,b] if rng.random()<.5 else [b,a])
    return out

class Vocab:
    def __init__(self,texts,max_vocab=10000):
        c=Counter()
        for t in texts:c.update(tokenize(t))
        self.itos=['<pad>','<unk>']+[w for w,_ in c.most_common(max_vocab-2)]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,s):return [self.stoi.get(w,1) for w in tokenize(s)]

class TinySSM(nn.Module):
    def __init__(self,V,emb=36,hid=56):
        super().__init__();self.emb=nn.Embedding(V,emb);self.rnn=nn.RNN(emb,hid,nonlinearity='tanh',batch_first=True);self.out=nn.Linear(hid,V)
    def forward(self,x,h=None):z,h=self.rnn(self.emb(x),h);return self.out(z),h

def dev(model):return next(model.parameters()).device

def train_sentence(model,opt,ids):
    if len(ids)<2:return np.nan
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    opt.zero_grad(set_to_none=True);logits,_=model(x);loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1));loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();return float(loss.item())

def seq_loss(model,ids):
    if len(ids)<2:return 0.
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return float(nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)).item())

def continuation_loss(model,vocab,prefix,target):
    """Cross-entropy on target tokens only, after conditioning on prefix."""
    pids=vocab.encode(prefix);tids=vocab.encode(target)
    if not pids or not tids:return 0.0
    ids=pids+tids;d=dev(model)
    x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    start=max(0,len(pids)-1);end=min(logits.shape[1],start+len(tids))
    if end<=start:return 0.0
    return float(nn.functional.cross_entropy(logits[:,start:end,:].reshape(-1,logits.size(-1)),y[:,start:end].reshape(-1)).item())

def logits_for_prompt(model,vocab,prompt):
    ids=vocab.encode(prompt);d=dev(model);x=torch.tensor(ids,dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return logits[0,-1]

def token_rank(model,vocab,prompt,answer):
    v=logits_for_prompt(model,vocab,prompt);aid=vocab.stoi.get(answer,1);return int((v>v[aid]).sum().item())+1

def top_tokens(model,vocab,prompt,k=5):
    v=logits_for_prompt(model,vocab,prompt);return [vocab.itos[i] for i in torch.topk(v,k=min(k,len(vocab.itos))).indices.tolist()]

def candidate_rank(model,vocab,prompt,true_answer,candidates):
    v=logits_for_prompt(model,vocab,prompt)
    scores=[]
    for a in candidates:
        scores.append((a,float(v[vocab.stoi.get(a,1)].item())))
    scores.sort(key=lambda x:x[1],reverse=True)
    order=[a for a,_ in scores]
    return order.index(true_answer)+1,dict(scores)

# -----------------------------------------------------------------------------
# Candidate evidence and v3.21-style dependency/factor inference
# -----------------------------------------------------------------------------

def unique_candidate_table(train,vocab,early_model,mid_model,final_model):
    """Build candidate evidence from observed consequences, without semantic labels."""
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());N=len(train)
    df=Counter()
    for s in docs:df.update(set(content_tokens(s)))
    token_sets=[set(content_tokens(s)) for s in train]
    rows=[]
    for idx,s in enumerate(docs):
        toks=content_tokens(s);st=set(toks);pp=pos[s];first=min(pp);last=max(pp)
        recurrence=math.log1p(len(pp))
        token_persistence=float(np.mean([math.log1p(df[t]) for t in toks])) if toks else 0.
        cross_context=float(np.mean([df[t]>=3 for t in toks])) if toks else 0.
        temporal_span=(last-first)/max(1,N-1) if len(pp)>1 else 0.
        early_pred=-seq_loss(early_model,vocab.encode(s));mid_pred=-seq_loss(mid_model,vocab.encode(s));final_pred=-seq_loss(final_model,vocab.encode(s))
        gain_early_mid=mid_pred-early_pred;gain_mid_final=final_pred-mid_pred

        future_idx=list(range(first+1,N));supports=[]
        if st and future_idx:
            for w in np.array_split(np.asarray(future_idx,dtype=int),3):
                seen=set()
                for j in w.tolist():seen |= token_sets[j]
                supports.append(len(st&seen)/max(1,len(st)))
        future_support=float(np.mean(supports)) if supports else 0.

        maxjac=0.;seen_prev=set()
        if st:
            for j in range(first):
                if train[j] in seen_prev:continue
                seen_prev.add(train[j]);ps=token_sets[j]
                if ps:maxjac=max(maxjac,len(st&ps)/(len(st|ps)+1e-12))
        novelty=1.-maxjac

        rel=[];unrel=[]
        for j in range(first+1,N):
            if train[j]==s:continue
            ov=len(st&token_sets[j]) if st else 0
            if ov>0:rel.append((ov/(len(st|token_sets[j])+1e-12),j))
            elif len(unrel)<6:unrel.append(j)
        rg=[]
        for _,j in sorted(rel,reverse=True)[:2]:
            t=train[j];rg.append(seq_loss(final_model,vocab.encode(t))-continuation_loss(final_model,vocab,s,t))
        context_gain=float(np.mean(rg)) if rg else 0.
        ug=[]
        for j in unrel[:2]:
            t=train[j];ug.append(continuation_loss(final_model,vocab,s,t)-seq_loss(final_model,vocab.encode(t)))
        low_interference=-float(np.mean(ug)) if ug else 0.

        rows.append({'uid':idx,'text':s,'first_pos':first,'last_pos':last,'count':len(pp),
                     'recurrence':recurrence,'token_persistence':token_persistence,'cross_context':cross_context,
                     'temporal_span':temporal_span,'early_predictability':early_pred,'mid_predictability':mid_pred,'future_predictability':final_pred,
                     'gain_early_mid':gain_early_mid,'gain_mid_final':gain_mid_final,'future_support':future_support,
                     'context_gain':context_gain,'novelty':novelty,'low_interference':low_interference})
    return pd.DataFrame(rows)

SOURCE_COLS=['recurrence','token_persistence','cross_context','temporal_span','early_predictability']
CONSEQUENCE_COLS=['gain_early_mid','gain_mid_final','future_support','context_gain']

def zcols(X):
    X=np.asarray(X,float);mu=X.mean(0);sd=X.std(0);active=sd>1e-8;Z=np.zeros_like(X)
    Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)
    return Z,mu,sd,active

def connected_components_from_corr(C,threshold=.72):
    n=C.shape[0];adj={i:set() for i in range(n)}
    for i in range(n):
        for j in range(i+1,n):
            if C[i,j]>threshold:adj[i].add(j);adj[j].add(i)
    comps=[];seen=set()
    for i in range(n):
        if i in seen:continue
        st=[i];c=set()
        while st:
            x=st.pop()
            if x in seen:continue
            seen.add(x);c.add(x);st.extend(adj[x]-seen)
        if len(c)>=2:comps.append(c)
    return comps

def learn_multifactor(cov,maxf=3):
    n=cov.shape[0];vals,vecs=np.linalg.eigh(cov);o=np.argsort(vals)[::-1];vals=vals[o];vecs=vecs[:,o]
    base=np.median(vals[min(maxf,n):])+1e-9 if n>maxf else 1e-9
    groups=[];loads=[]
    for k in range(min(maxf,n)):
        strength=max(0.,float((vals[k]-base)/(vals[k]+1e-9)))
        v=np.abs(vecs[:,k]);mx=v.max()+1e-12;g={i for i,l in enumerate(v) if l/mx>.58}
        if len(g)>=2 and strength>.22 and not any(len(g&h)/len(g|h)>.75 for h in groups):
            groups.append(g);loads.append((v,strength))
    return groups,loads,vals

def cap_weights(raw,g):
    g=[i for i in g if raw[i]>0]
    if len(g)<=1:return raw
    total=sum(raw[i] for i in g);cap=max(raw[i] for i in g);f=cap/(total+1e-12)
    for i in g:raw[i]*=f
    return raw

def evidence_model(cand,mode='factor'):
    X=cand[SOURCE_COLS].to_numpy(float);Z,mu,sd,active=zcols(X)
    y=cand['future_predictability'].to_numpy(float);yz=(y-y.mean())/(y.std()+1e-9)
    residual=Z-yz[:,None]
    bias=residual.mean(0)
    rho=np.mean(np.exp(-np.abs(residual)/.8),axis=0)
    cov=np.cov((residual-residual.mean(0)).T)+np.eye(len(SOURCE_COLS))*1e-6
    rsd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(rsd[:,None]*rsd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72)
    groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:float(rho[i]**3) if active[i] else 0. for i in range(len(SOURCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(SOURCE_COLS))]);w/=w.sum()+1e-12
    deb=Z-bias[None,:];U=deb@w
    disagreement=np.sum(w[None,:]*(deb-U[:,None])**2,axis=1)
    # v2-style common-mode risk: don't 'correct' value; lower confidence when many channels miss together.
    substantial=np.abs(residual)>.8
    same_sign=np.maximum((residual>.8).sum(1),(residual<-.8).sum(1))
    h=float(np.mean(same_sign>=max(2,int(math.ceil(len(SOURCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h))
    U01=1/(1+np.exp(-np.clip(U,-12,12)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf;out['future_yz']=yz
    meta={'source_cols':SOURCE_COLS,'rho':rho.tolist(),'bias':bias.tolist(),'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[SOURCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[SOURCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

def rank_centered(x):
    x=np.asarray(x,float);r=pd.Series(x).rank(method='average',pct=True).to_numpy(float)
    return 2.*(r-.5)

def multi_consequence_model(cand,mode='factor'):
    """Dependency-discount multiple directly observed consequence channels."""
    Y=np.column_stack([rank_centered(cand[c].to_numpy(float)) for c in CONSEQUENCE_COLS])
    active=np.std(Y,axis=0)>1e-8
    cov=np.cov(Y.T)+np.eye(len(CONSEQUENCE_COLS))*1e-6
    sd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(sd[:,None]*sd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72);groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:(1. if active[i] else 0.) for i in range(len(CONSEQUENCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(CONSEQUENCE_COLS))],float);w/=w.sum()+1e-12
    U=Y@w;disagreement=np.sum(w[None,:]*(Y-U[:,None])**2,axis=1)
    same_bad=(Y<-.40).sum(1);h=float(np.mean(same_bad>=max(2,int(math.ceil(len(CONSEQUENCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h));U01=1/(1+np.exp(-2*np.clip(U,-6,6)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf
    # Keep novelty and interference structurally distinct from consequence value.
    # Novelty can soften a replacement wall, but it cannot make a bad candidate good.
    # Interference is a cost/constraint, not a positive evidence vote.
    out['novelty01']=(rank_centered(out['novelty'].to_numpy(float))+1.)/2.
    out['interference_risk01']=(rank_centered(-out['low_interference'].to_numpy(float))+1.)/2.
    meta={'consequence_cols':CONSEQUENCE_COLS,'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[CONSEQUENCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[CONSEQUENCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

def portfolio_model(cand_v43,cand_multi):
    """Keep alternative consequence models as parallel support lanes.

    A candidate may earn permanence from either lane; one scalar model is not
    allowed to erase a structurally different useful profile. No semantic labels
    define the lanes: one is the prior predictive structural estimator, the other
    is direct multi-consequence evidence.
    """
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    out=b.copy();ua=a.utility01.to_numpy(float);ub=b.utility01.to_numpy(float);qa=a.evidence_confidence.to_numpy(float);qb=b.evidence_confidence.to_numpy(float)
    choose_b=ub>=ua
    out['utility_v43']=ua;out['utility_multi']=ub;out['confidence_v43']=qa;out['confidence_multi']=qb
    out['utility01']=np.maximum(ua,ub);out['evidence_confidence']=np.where(choose_b,qb,qa);out['portfolio_lane']=np.where(choose_b,'multi','v43')
    return out

# -----------------------------------------------------------------------------
# Parity-protected raw templates + comparative replacement bank
# Address != template: address is transparent IDF token signature; template is raw ids.
# -----------------------------------------------------------------------------

class Slot:
    PRIME=1000003
    def __init__(self,text,ids,uid,U,S,Q,count,first_pos,created_step):
        self.text=text;self.a=list(ids);self.b=list(ids);self.uid=uid;self.U=float(U);self.S=float(S);self.Q=float(Q);self.count=int(count);self.first_pos=int(first_pos);self.created_step=int(created_step)
        self.p,self.q=self.chk(self.a);self.corrupted=False
    @classmethod
    def chk(cls,ids):
        q=np.asarray(ids,dtype=np.int64);return int(q.sum()%cls.PRIME),int(((np.arange(len(q))+1)*q).sum()%cls.PRIME)
    def corrupt(self,rng):
        if self.a:
            j=rng.randrange(len(self.a));self.a[j]=(self.a[j]+rng.randint(1,19))%100000;self.corrupted=True
    def retrieve(self,parity=True):
        if self.chk(self.a)==(self.p,self.q):return list(self.a),'primary'
        if parity and self.chk(self.b)==(self.p,self.q):return list(self.b),'repaired'
        return None,'lost'

class SemanticBank:
    def __init__(self,capacity,vocab,idf,comparative=True,mode='factor'):
        self.capacity=capacity;self.vocab=vocab;self.idf=idf;self.comparative=comparative;self.mode=mode;self.slots=[];self.writes=0;self.replacements=0;self.blocked=0;self.holds=0;self.margin=0.
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return slot.U*math.sqrt(max(1e-9,math.log1p(slot.count)))/(1+.15*age)
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.stability)*(.75+.25*row.evidence_confidence))
    def admit(self,row,step,margin):
        if row.evidence_confidence <= 0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals))
        if self.comparative:
            if c-vals[j] <= margin:self.blocked+=1;return False
        else:
            # Old/automatic full-bank behavior: qualifying candidate evicts weakest.
            if c<.5:self.blocked+=1;return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;return True
    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for s in self.slots:
            if rng.random()<p:s.corrupt(rng)
    def replay_ids(self,parity=True):
        out=[];repair=lost=0
        for s in self.slots:
            ids,status=s.retrieve(parity)
            if ids is not None:out.append(ids)
            repair+=status=='repaired';lost+=status=='lost'
        return out,repair,lost
    def query_score(self,q_tokens,slot):
        q=set(q_tokens);s=set(content_tokens(slot.text));ov=q&s
        if not ov:return 0.
        num=sum(self.idf.get(t,1.) for t in ov);den=sum(self.idf.get(t,1.) for t in q)+1e-12
        return float(num/den)
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        # Honest associative retrieval: never force a guess on tied/no-overlap addresses.
        q=content_tokens(prompt);chosen=[];seen=set();logs=[]
        for hop in range(max_hops):
            scores=[]
            for i,s in enumerate(self.slots):
                if i in seen:continue
                scores.append((self.query_score(q,s),i))
            if not scores:break
            scores.sort(reverse=True);best,bi=scores[0];second=scores[1][0] if len(scores)>1 else 0.
            # Dynamic margin: rare-token exact overlap should separate from the next address.
            margin=max(.025,.10*best)
            if best<=0 or best-second<=margin:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin});break
            slot=self.slots[bi];ids,status=slot.retrieve(parity)
            if ids is None:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin,'status':'lost'});break
            chosen.append(slot.text);seen.add(bi);q += content_tokens(slot.text)
            logs.append({'hop':hop,'resolved':True,'best':best,'second':second,'margin':margin,'status':status,'text':slot.text,'uid':slot.uid})
        return chosen,logs


class AdaptiveSemanticBank(SemanticBank):
    """Bounded soft-wall replacement: churn stiffens; persistent blocking softens."""
    def __init__(self,capacity,vocab,idf,comparative=True,adaptive=True,mode='v44'):
        super().__init__(capacity,vocab,idf,comparative,mode)
        self.adaptive=adaptive;self.base_margin=0.;self.current_margin=0.;self.margin_min=np.inf;self.margin_max=0.
        self.replace_ema=0.;self.block_ema=0.;self.pressure_alpha=.08
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.evidence_confidence)*(.75+.25*(1.-float(getattr(row,'interference_risk01',0.)))))
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return float(slot.U*(.90+.10*slot.Q)/(1+.08*age))
    def _pressure(self,event):
        a=self.pressure_alpha
        self.replace_ema=(1-a)*self.replace_ema+a*(1. if event=='replace' else 0.)
        self.block_ema=(1-a)*self.block_ema+a*(1. if event=='block' else 0.)
        if self.adaptive:
            scale=float(np.clip(math.exp(1.25*(self.replace_ema-self.block_ema)),.25,2.5))
            self.current_margin=self.base_margin*scale
        self.margin_min=min(self.margin_min,self.current_margin);self.margin_max=max(self.margin_max,self.current_margin)
    def admit(self,row,step,margin=None):
        if row.evidence_confidence<=0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals));m=self.current_margin if self.comparative else 0.
        # Novelty is pressure for admission, not utility. It may reduce the extra
        # replacement margin, but never allows a lower-value candidate to win.
        novelty=float(getattr(row,'novelty01',0.));m_eff=m*(1.-.45*novelty)
        if self.comparative and c-vals[j]<=m_eff:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

def multi_gate_margin(cand):
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.evidence_confidence.to_numpy(float))*(.75+.25*(1.-cand.interference_risk01.to_numpy(float)))
    q25,q75=np.quantile(C,[.25,.75]);return float(.18*max(1e-6,q75-q25))

def build_v44_bank(cand,vocab,kind='multi_adaptive_gate',capacity=96):
    idf=compute_idf(cand);comparative=('auto' not in kind);adaptive=('adaptive' in kind)
    bank=AdaptiveSemanticBank(capacity,vocab,idf,comparative=comparative,adaptive=adaptive,mode=kind)
    base=multi_gate_margin(cand);bank.base_margin=base;bank.current_margin=base;bank.margin=base;bank.margin_min=base;bank.margin_max=base
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,base)
    bank.margin=bank.current_margin
    return bank

class DualLaneBank(AdaptiveSemanticBank):
    """Shared-capacity bank that preserves emergent evidence lanes without semantic quotas."""
    def __init__(self,capacity,vocab,idf,mode='dual_lane_adaptive'):
        super().__init__(capacity,vocab,idf,comparative=True,adaptive=True,mode=mode)
        self.lane_pressure=Counter()
    def _lane(self,row):
        return str(getattr(row,'portfolio_lane','single'))
    def admit(self,row,step,margin=None):
        if row.evidence_confidence<=0 or row.utility01<.5:
            self.holds+=1;return False
        lane=self._lane(row);self.lane_pressure[lane]+=1
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step);slot.lane=lane
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        lanes=set(self.lane_pressure)|{getattr(x,'lane','single') for x in self.slots}
        total=max(1,sum(self.lane_pressure.values()))
        desired={l:max(1,int(round(self.capacity*self.lane_pressure[l]/total))) for l in lanes}
        # Repair rounding to capacity deterministically.
        while sum(desired.values())>self.capacity:
            l=max(sorted(desired),key=lambda x:desired[x]);desired[l]-=1
        while sum(desired.values())<self.capacity:
            l=max(sorted(desired),key=lambda x:self.lane_pressure[x]);desired[l]+=1
        counts=Counter(getattr(x,'lane','single') for x in self.slots)
        if counts[lane] < desired.get(lane,0):
            over=[l for l in lanes if counts[l]>desired.get(l,0)]
            pool=[i for i,x in enumerate(self.slots) if getattr(x,'lane','single') in over] if over else list(range(len(self.slots)))
        else:
            pool=[i for i,x in enumerate(self.slots) if getattr(x,'lane','single')==lane]
            if not pool:pool=list(range(len(self.slots)))
        vals=[self.retained_value(self.slots[i],step) for i in pool];pi=int(np.argmin(vals));j=pool[pi];weak=vals[pi]
        m=self.current_margin;novelty=float(getattr(row,'novelty01',0.));m_eff=m*(1.-.45*novelty)
        if c-weak<=m_eff:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

def build_dual_lane_bank(cand,vocab,capacity=96):
    idf=compute_idf(cand);bank=DualLaneBank(capacity,vocab,idf)
    base=multi_gate_margin(cand);bank.base_margin=base;bank.current_margin=base;bank.margin=base;bank.margin_min=base;bank.margin_max=base
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,base)
    bank.margin=bank.current_margin
    return bank

class SplitRoleBank:
    """One total memory budget, two learned operational roles.

    Replay/consolidation and query-time addressing compete under separate half-budget
    banks because the local ablation showed they optimize different consequences.
    This 50/50 split is an explicit scaffold and is logged for later adaptive removal.
    """
    def __init__(self,replay_bank,query_bank,total_capacity):
        self.replay_bank=replay_bank;self.query_bank=query_bank;self.capacity=total_capacity;self.mode='split_role'
    @property
    def slots(self):return self.query_bank.slots
    @property
    def writes(self):return self.replay_bank.writes+self.query_bank.writes
    @property
    def replacements(self):return self.replay_bank.replacements+self.query_bank.replacements
    @property
    def blocked(self):return self.replay_bank.blocked+self.query_bank.blocked
    @property
    def holds(self):return self.replay_bank.holds+self.query_bank.holds
    @property
    def margin(self):return (getattr(self.replay_bank,'margin',0.)+getattr(self.query_bank,'margin',0.))/2
    @property
    def margin_min(self):return min(getattr(self.replay_bank,'margin_min',self.margin),getattr(self.query_bank,'margin_min',self.margin))
    @property
    def margin_max(self):return max(getattr(self.replay_bank,'margin_max',self.margin),getattr(self.query_bank,'margin_max',self.margin))
    @property
    def replace_ema(self):return getattr(self.replay_bank,'replace_ema',0.)+getattr(self.query_bank,'replace_ema',0.)
    @property
    def block_ema(self):return getattr(self.replay_bank,'block_ema',0.)+getattr(self.query_bank,'block_ema',0.)
    def corrupt(self,p,seed):
        self.replay_bank.corrupt(p,seed);self.query_bank.corrupt(p,seed+100003)
    def replay_ids(self,parity=True):
        a,ra,la=self.replay_bank.replay_ids(parity);_,rb,lb=self.query_bank.replay_ids(parity);return a,ra+rb,la+lb
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        return self.query_bank.retrieve_templates(prompt,max_hops,parity)

def build_split_role_bank(cand_v43,cand_multi,vocab,capacity=96):
    replay_cap=capacity//2;query_cap=capacity-replay_cap
    replay=build_v44_bank(cand_multi,vocab,'multi_auto',replay_cap)
    query=build_bank(cand_v43,{},vocab,'factor_gate',query_cap)
    return SplitRoleBank(replay,query,capacity)

def compute_idf(cand):
    docs=[set(content_tokens(s)) for s in cand.text];df=Counter();
    for d in docs:df.update(d)
    N=len(docs);return {t:(math.log((N+1)/(c+1))+1)**2 for t,c in df.items()}

def add_stability(cand):
    x=cand.copy();span=x.temporal_span.to_numpy(float);cnt=np.log1p(x['count'].to_numpy(float));cnt=cnt/(cnt.max()+1e-12);x['stability']=np.clip(.5*span+.5*cnt,0,1);return x

def gate_margin(cand):
    # Field report says the margin is calibration-dependent, not universal.
    # Here it scales to candidate-value dispersion using no semantic labels.
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.stability.to_numpy(float))*(.75+.25*cand.evidence_confidence.to_numpy(float))
    return float(.25*np.std(C))

def build_bank(cand,meta,vocab,kind='factor_gate',capacity=96):
    idf=compute_idf(cand);bank=SemanticBank(capacity,vocab,idf,comparative=('auto' not in kind),mode=kind)
    margin=gate_margin(cand);bank.margin=margin
    # Process in observational order, not best-first.
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,margin)
    return bank

def old_v42_bank(train,vocab,final_model,capacity=96):
    # v4.2 selector, but bank is unique-template bounded for fair address retrieval.
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());counts=Counter();toks=[]
    for s in docs:
        q=content_tokens(s);toks.append(q);counts.update(set(q))
    rows=[]
    for i,(s,q) in enumerate(zip(docs,toks)):
        rows.append({'uid':i,'text':s,'first_pos':min(pos[s]),'count':len(pos[s]),
                     'recurrence':np.mean([math.log1p(counts[w]) for w in q]) if q else 0.,
                     'cross_context':np.mean([counts[w]>=3 for w in q]) if q else 0.,
                     'predictability':-seq_loss(final_model,vocab.encode(s))})
    d=pd.DataFrame(rows);X=d[['recurrence','cross_context','predictability']].to_numpy(float);Z,_,_,active=zcols(X)
    C=np.zeros((3,3));idx=np.where(active)[0]
    if len(idx)>1:
        Q=np.nan_to_num(np.abs(np.corrcoef(Z[:,idx],rowvar=False)));np.fill_diagonal(Q,0)
        for a,i in enumerate(idx):
            for b,j in enumerate(idx):C[i,j]=Q[a,b]
    rho=np.where(active,.7,0);a=rho**3;base=a/(a.sum()+1e-12);w=a/(1+1.5*(C@base));w/=w.sum()+1e-12;score=Z@w
    sets=[set(content_tokens(s)) for s in docs];selected=[];remain=set(range(len(d)))
    while remain and len(selected)<min(capacity,len(d)):
        best=None;bv=-1e99
        for i in remain:
            red=max((len(sets[i]&sets[j])/(len(sets[i]|sets[j])+1e-12) for j in selected),default=0.);v=float(score[i])-.85*red
            if v>bv:bv=v;best=i
        selected.append(best);remain.remove(best)
    # Reuse bank container, direct add selected templates.
    idf=compute_idf(pd.DataFrame({'text':docs}));bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='v42_selector')
    for step,i in enumerate(selected):
        row=d.iloc[i];slot=Slot(row.text,vocab.encode(row.text),int(row.uid),.75,.75,.75,int(row['count']),int(row.first_pos),step);bank.slots.append(slot);bank.writes+=1
    return bank

def random_bank(train,vocab,capacity,seed):
    docs=list(dict.fromkeys(train));rng=random.Random(seed);picks=rng.sample(docs,min(capacity,len(docs)));tmp=pd.DataFrame({'text':docs});idf=compute_idf(tmp);bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='random')
    pos={s:i for i,s in enumerate(docs)}
    for step,s in enumerate(picks):bank.slots.append(Slot(s,vocab.encode(s),pos[s],.5,.5,.5,1,pos[s],step));bank.writes+=1
    return bank

# -----------------------------------------------------------------------------
# Binding-specific probes. No chain ids/answers enter training or memory selection.
# -----------------------------------------------------------------------------

def augment_prompt(bank,prompt,parity=True,order='path'):
    if bank is None:return prompt,[],[]
    templates,logs=bank.retrieve_templates(prompt,2,parity)
    if order=='reverse':templates=list(reversed(templates))
    aug=(' '.join(templates)+' '+prompt).strip() if templates else prompt
    return aug,templates,logs

def probe_panel(model,vocab,chains,seed,stage,variant,bank=None,parity=True):
    rows=[]
    def add(kind,prompt,ans,item,mode='plain',candidates=None,wrong=None):
        aug=prompt;retr=[];logs=[]
        if mode in ('assist','assist_reverse') and bank is not None:
            aug,retr,logs=augment_prompt(bank,prompt,parity,'reverse' if mode=='assist_reverse' else 'path')
        elif mode in ('first_only','second_only') and bank is not None:
            # Single-premise controls separate retrieval from composition.
            _,rr,ll=augment_prompt(bank,prompt,parity,'path');retr=(rr[:1] if mode=='first_only' else rr[-1:]) if rr else [];logs=ll;aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        r=token_rank(model,vocab,aug,ans);cr=np.nan;margin=np.nan;wrong_score=np.nan;true_score=np.nan
        if candidates:
            cr,sc=candidate_rank(model,vocab,aug,ans,candidates);true_score=sc.get(ans,np.nan)
            if wrong is not None:wrong_score=sc.get(wrong,np.nan);margin=true_score-wrong_score
        rows.append({'seed':seed,'stage':stage,'variant':variant,'probe':kind,'mode':mode,'item':item,'prompt':prompt,'aug_prompt':aug,'answer':ans,
                     'rank':r,'candidate_rank':cr,'binding_margin':margin,'true_score':true_score,'wrong_score':wrong_score,
                     'top1':int(r==1),'top5':int(r<=5),'candidate_top1':int(cr==1) if not np.isnan(cr) else np.nan,
                     'top_tokens':'|'.join(top_tokens(model,vocab,aug,5)),'retrieved':' || '.join(retr),'retrieval_log':json.dumps(logs)})
    for i,(p,a) in enumerate(FACTS):add('exact_fact',p,a,f'fact{i}')
    for i,(p,a) in enumerate(REPHRASE):add('rephrase',p,a,f'rephrase{i}')
    q=chains[chains.kind=='probe'].reset_index(drop=True);zcands=q.z.tolist();xcands=q.x.tolist()
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        for mode in ['plain','assist','assist_reverse','first_only','second_only']:
            add('two_hop',f'{row.x} ultimately maps to',row.z,f'chain{i}',mode,zcands,wrong.z)
        add('two_hop_shuffled',f'{row.x} ultimately maps to',wrong.z,f'chain{i}','plain',zcands,row.z)
        for mode in ['plain','assist']:
            add('inverse',f'{row.y} is contained by',row.x,f'chain{i}',mode,xcands,wrong.x)
        add('premise_contains',f'{row.x} contains',row.y,f'chain{i}')
        add('premise_maps',f'{row.y} maps to',row.z,f'chain{i}')
    return pd.DataFrame(rows)

def summarize_probes(df):
    def safe_cmrr(x):
        a=np.asarray(x,float);a=a[np.isfinite(a)]
        return float(np.mean(1/a)) if len(a) else np.nan
    return df.groupby(['stage','variant','probe','mode']).agg(top1=('top1','mean'),top5=('top5','mean'),candidate_top1=('candidate_top1','mean'),
        mrr=('rank',lambda x:float(np.mean(1/np.asarray(x,float)))),candidate_mrr=('candidate_rank',safe_cmrr),
        binding_margin=('binding_margin','mean'),median_rank=('rank','median'),n=('rank','size')).reset_index()


# =============================================================================
# Adrianic v4.5 — Multi-access memory objects + explicit identity-binding state
# =============================================================================
OUT = ROOT / 'adrianic_v45_multiaccess_binding_results'
OUT.mkdir(parents=True, exist_ok=True)

class MultiAccessSlot(Slot):
    def __init__(self,text,ids,uid,C,R,QC,QR,count,first_pos,created_step):
        super().__init__(text,ids,uid,U=max(float(C),float(R)),S=.5,Q=max(float(QC),float(QR)),
                         count=count,first_pos=first_pos,created_step=created_step)
        self.C=float(C); self.R=float(R); self.QC=float(QC); self.QR=float(QR)

class MultiAccessBank:
    # Confidence remains logged and can HOLD truly unresolved candidates, but it is
    # not allowed to re-multiply an already evidence-adjusted access value into near-zero.
    # The first v4.5 screen showed that doing so destroyed retrieval coverage.
    def __init__(self,capacity,vocab,idf,adaptive=True):
        self.capacity=capacity;self.vocab=vocab;self.idf=idf;self.adaptive=adaptive
        self.slots=[];self.writes=0;self.replacements=0;self.blocked=0;self.holds=0
        self.base_margin=0.;self.current_margin=0.;self.margin=0.
        self.margin_min=np.inf;self.margin_max=0.;self.replace_ema=0.;self.block_ema=0.;self.alpha=.08

    @staticmethod
    def survival(C,R,QC,QR):
        c=float(C)*(.90+.10*float(QC));r=float(R)*(.90+.10*float(QR))
        return max(c,r)

    def _pressure(self,event):
        a=self.alpha
        self.replace_ema=(1-a)*self.replace_ema+a*float(event=='replace')
        self.block_ema=(1-a)*self.block_ema+a*float(event=='block')
        if self.adaptive:
            scale=float(np.clip(math.exp(.85*(self.replace_ema-self.block_ema)),.35,2.0))
            self.current_margin=self.base_margin*scale
        self.margin_min=min(self.margin_min,self.current_margin)
        self.margin_max=max(self.margin_max,self.current_margin)

    def admit(self,row,step):
        C=float(row.C);R=float(row.R);QC=float(row.QC);QR=float(row.QR)
        if (QC<=0 and QR<=0) or max(C,R)<.50:
            self.holds+=1;return False
        slot=MultiAccessSlot(row.text,self.vocab.encode(row.text),int(row.uid),C,R,QC,QR,
                             int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=np.array([self.survival(s.C,s.R,s.QC,s.QR) for s in self.slots])
        j=int(np.argmin(vals));cand=self.survival(C,R,QC,QR)
        nov=float(getattr(row,'novelty01',0.));m=self.current_margin*(1-.35*nov)
        if cand<=vals[j]+m:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for s in self.slots:
            if rng.random()<p:s.corrupt(rng)

    def replay_ids(self,parity=True):
        ranked=sorted(self.slots,key=lambda s:(s.C*(.90+.10*s.QC),s.created_step),reverse=True)
        out=[];repair=lost=0
        for s in ranked:
            ids,status=s.retrieve(parity)
            if ids is not None:out.append(ids)
            repair+=status=='repaired';lost+=status=='lost'
        return out,repair,lost

    def _address_score(self,q_tokens,slot):
        q=set(q_tokens);st=set(content_tokens(slot.text));ov=q&st
        if not ov:return 0.
        num=sum(self.idf.get(t,1.) for t in ov);den=sum(self.idf.get(t,1.) for t in q)+1e-12
        return float(num/den)*(.30+.70*slot.R)*(.90+.10*slot.QR)

    def retrieve_templates(self,prompt,max_hops=3,parity=True):
        q=content_tokens(prompt);chosen=[];seen=set();logs=[]
        for hop in range(max_hops):
            scores=[(self._address_score(q,s),i) for i,s in enumerate(self.slots) if i not in seen]
            if not scores:break
            scores.sort(reverse=True);best,bi=scores[0];second=scores[1][0] if len(scores)>1 else 0.
            margin=max(.018,.07*best)
            if best<=0 or best-second<=margin:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin});break
            s=self.slots[bi];ids,status=s.retrieve(parity)
            if ids is None:break
            chosen.append(s.text);seen.add(bi);q+=content_tokens(s.text)
            logs.append({'hop':hop,'resolved':True,'uid':s.uid,'R':s.R,'C':s.C,'text':s.text,'status':status})
        return chosen,logs

    def identity_join(self,prompt,parity=True,shuffle=False):
        q=content_tokens(prompt)
        if not q:return {},[],{'resolved':False,'reason':'empty_query'}
        anchors=sorted(set(q),key=lambda t:self.idf.get(t,1.),reverse=True)[:max(1,min(2,len(set(q))))]
        first=[]
        for i,s in enumerate(self.slots):
            toks=content_tokens(s.text);aset=set(toks)&set(anchors)
            if aset:first.append((self._address_score(anchors,s),i,aset,toks))
        first.sort(reverse=True)
        if not first:return {},[],{'resolved':False,'reason':'no_anchor_template'}

        path_scores=defaultdict(float);paths=[]
        for fs,fi,aset,atoks in first[:8]:
            A=self.slots[fi];apos={t:j for j,t in enumerate(atoks)}
            bridges=sorted(
                [t for t in set(atoks) if t not in set(q) and self.idf.get(t,1.)>1.05],
                key=lambda t:self.idf.get(t,1.),reverse=True
            )[:8]
            for bridge in bridges:
                valid_anchor=[a for a in aset if apos.get(a,10**9)<apos.get(bridge,-1)]
                if not valid_anchor:continue
                for bi,B in enumerate(self.slots):
                    if bi==fi:continue
                    btoks=content_tokens(B.text)
                    if bridge not in btoks:continue
                    bpos={t:j for j,t in enumerate(btoks)};brp=bpos[bridge]
                    candidates=[t for t in btoks[brp+1:] if t not in set(q) and t!=bridge and self.idf.get(t,1.)>1.05]
                    if not candidates:continue
                    Baddr=(.30+.70*B.R)*(.90+.10*B.QR)
                    bridge_w=min(3.0,self.idf.get(bridge,1.))
                    anchor_w=max(min(3.0,self.idf.get(a,1.)) for a in valid_anchor)
                    for c in set(candidates):
                        dist=max(1,bpos[c]-brp);sc=fs*Baddr*bridge_w*anchor_w/dist
                        path_scores[c]+=float(sc)
                        paths.append({'A':A.text,'B':B.text,'bridge':bridge,'candidate':c,'score':float(sc)})
        if not path_scores:return {},[],{'resolved':False,'reason':'no_identity_path'}
        if shuffle:
            keys=sorted(path_scores);vals=[path_scores[k] for k in keys]
            rng=random.Random(int(hashlib.sha256(prompt.encode()).hexdigest()[:12],16));rng.shuffle(vals)
            path_scores=defaultdict(float,{k:v for k,v in zip(keys,vals)})
        mx=max(path_scores.values())+1e-12
        norm={k:float(v/mx) for k,v in path_scores.items()}
        return norm,sorted(paths,key=lambda x:x['score'],reverse=True)[:12],{'resolved':True,'anchors':anchors,'n_paths':len(paths)}

def build_multi_access_bank(cand_v43,cand_multi,vocab,capacity=96,adaptive=True):
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    merged=b.copy()
    merged['C']=b.utility01.to_numpy(float);merged['QC']=b.evidence_confidence.to_numpy(float)
    merged['R']=a.utility01.to_numpy(float);merged['QR']=a.evidence_confidence.to_numpy(float)
    idf=compute_idf(merged);bank=MultiAccessBank(capacity,vocab,idf,adaptive)
    sv=np.maximum(merged.C*(.90+.10*merged.QC),merged.R*(.90+.10*merged.QR))
    q25,q75=np.quantile(sv,[.25,.75]);bank.base_margin=.12*max(1e-6,q75-q25)
    bank.current_margin=bank.base_margin;bank.margin=bank.current_margin
    bank.margin_min=bank.current_margin;bank.margin_max=bank.current_margin
    for step,row in merged.sort_values('first_pos').iterrows():bank.admit(row,step)
    bank.margin=bank.current_margin
    return bank,merged


class ParetoMultiAccessBank(MultiAccessBank):
    """Online Pareto-retained multi-access bank.

    No fixed replay/retrieval quota and no scalar collapse of C and R.
    When full, candidate + stored points are ranked by nondominated fronts;
    crowding distance preserves useful extremes and removes redundant interior
    points. This is the standard resource geometry, not a semantic rule.
    """
    @staticmethod
    def _point_obj(obj):
        return np.array([
            obj.C*(.90+.10*obj.QC),
            obj.R*(.90+.10*obj.QR)
        ],float)

    @staticmethod
    def _fronts(points):
        n=len(points);dominated=[[] for _ in range(n)];count=np.zeros(n,int);front=[[]]
        for p in range(n):
            for q in range(n):
                if p==q:continue
                pp=points[p];qq=points[q]
                pdom=np.all(pp>=qq) and np.any(pp>qq)
                qdom=np.all(qq>=pp) and np.any(qq>pp)
                if pdom:dominated[p].append(q)
                elif qdom:count[p]+=1
            if count[p]==0:front[0].append(p)
        k=0
        while k<len(front) and front[k]:
            nxt=[]
            for p in front[k]:
                for q in dominated[p]:
                    count[q]-=1
                    if count[q]==0:nxt.append(q)
            k+=1
            if nxt:front.append(nxt)
        return front

    @staticmethod
    def _crowding(points,idxs):
        if not idxs:return {}
        if len(idxs)<=2:return {i:float('inf') for i in idxs}
        d={i:0. for i in idxs};P=np.asarray([points[i] for i in idxs])
        for dim in range(P.shape[1]):
            order=sorted(idxs,key=lambda i:points[i][dim])
            d[order[0]]=d[order[-1]]=float('inf')
            lo=points[order[0]][dim];hi=points[order[-1]][dim]
            if hi-lo<=1e-12:continue
            for j in range(1,len(order)-1):
                if np.isfinite(d[order[j]]):
                    d[order[j]]+=(points[order[j+1]][dim]-points[order[j-1]][dim])/(hi-lo)
        return d

    def admit(self,row,step):
        C=float(row.C);R=float(row.R);QC=float(row.QC);QR=float(row.QR)
        if (QC<=0 and QR<=0) or max(C,R)<.50:
            self.holds+=1;return False
        cand=MultiAccessSlot(row.text,self.vocab.encode(row.text),int(row.uid),C,R,QC,QR,
                             int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(cand);self.writes+=1;return True

        objs=self.slots+[cand];points=[self._point_obj(s) for s in objs]
        fronts=self._fronts(points)
        rank={}
        crowd={}
        for ri,f in enumerate(fronts):
            for i in f:rank[i]=ri
            crowd.update(self._crowding(points,f))
        # Worst = deepest Pareto front; within it, most redundant / least crowded.
        worst=max(range(len(objs)),key=lambda i:(rank.get(i,10**9),-crowd.get(i,0.)))
        if worst==len(objs)-1:
            self.blocked+=1;return False
        self.slots[worst]=cand;self.writes+=1;self.replacements+=1;return True

def build_pareto_multi_access_bank(cand_v43,cand_multi,vocab,capacity=96):
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    merged=b.copy();merged['C']=b.utility01.to_numpy(float);merged['QC']=b.evidence_confidence.to_numpy(float)
    merged['R']=a.utility01.to_numpy(float);merged['QR']=a.evidence_confidence.to_numpy(float)
    bank=ParetoMultiAccessBank(capacity,vocab,compute_idf(merged),adaptive=False)
    for step,row in merged.sort_values('first_pos').iterrows():bank.admit(row,step)
    return bank,merged


class IdentityBindingGraph:
    """Compact structural state derived from repeated ordered token identity.

    This is not a semantic parser. It never names subject/predicate/object roles.
    Repeated rare-token order creates directed edges; exact shared identity permits
    deterministic multi-step traversal.
    """
    def __init__(self,cand_v43,max_edges=512):
        self.idf=compute_idf(cand_v43)
        vals=np.array(list(self.idf.values()),float)
        self.rare_floor=float(np.quantile(vals,.60)) if len(vals) else 1.
        edge_weight=defaultdict(float);edge_support=defaultdict(float);edge_sources=defaultdict(int)
        for _,row in cand_v43.iterrows():
            toks=content_tokens(row.text)
            rare=[t for t in toks if self.idf.get(t,1.)>=self.rare_floor]
            # preserve order and remove repeated token occurrences inside one sentence
            clean=[]
            for t in rare:
                if not clean or clean[-1]!=t:clean.append(t)
            if len(clean)<2:continue
            base=float(row.utility01)*(.90+.10*float(row.evidence_confidence))*math.log1p(max(1,int(row['count'])))
            for i in range(len(clean)):
                for j in range(i+1,len(clean)):
                    a,b=clean[i],clean[j]
                    if a==b:continue
                    edge_weight[(a,b)]+=base
                    edge_support[(a,b)]+=int(row['count'])
                    edge_sources[(a,b)]+=1
        # Persistence gate: a structural edge must have recurring support.
        edges=[(k,w,edge_support[k],edge_sources[k]) for k,w in edge_weight.items() if edge_support[k]>=2]
        edges.sort(key=lambda x:(x[1],x[2]),reverse=True);edges=edges[:max_edges]
        self.edges={(a,b):float(w) for (a,b),w,_,_ in edges}
        self.support={(a,b):float(s) for (a,b),_,s,_ in edges}
        self.out=defaultdict(dict)
        for (a,b),w in self.edges.items():self.out[a][b]=w
        self.max_edges=max_edges

    def _anchor(self,prompt):
        q=content_tokens(prompt)
        if not q:return None
        return max(set(q),key=lambda t:self.idf.get(t,1.))

    def one_step(self,prompt):
        x=self._anchor(prompt)
        if x is None:return {},[]
        raw=dict(self.out.get(x,{}));mx=max(raw.values(),default=0.)+1e-12
        return ({k:float(v/mx) for k,v in raw.items()} if raw else {}),[{'x':x,'z':k,'score':v} for k,v in raw.items()]

    def two_step(self,prompt):
        x=self._anchor(prompt)
        if x is None:return {},[]
        raw=defaultdict(float);paths=[]
        for y,w1 in self.out.get(x,{}).items():
            for z,w2 in self.out.get(y,{}).items():
                if z==x:continue
                # geometric mean prevents one huge edge from overwhelming a weak second edge.
                s=math.sqrt(max(0.,w1*w2))
                raw[z]+=s
                paths.append({'x':x,'bridge':y,'z':z,'score':float(s),'w1':float(w1),'w2':float(w2)})
        mx=max(raw.values(),default=0.)+1e-12
        return ({k:float(v/mx) for k,v in raw.items()} if raw else {}),sorted(paths,key=lambda d:d['score'],reverse=True)[:12]

    def shuffled_two_step_mean(self,prompt,n=32):
        # Strong negative control: preserve first-hop edges and the full collection
        # of second-hop adjacency lists, but break the exact bridge identity by
        # permuting which source-node adjacency list is consulted at hop two.
        x=self._anchor(prompt)
        if x is None or x not in self.out:return {}
        source_nodes=sorted(self.out.keys())
        if len(source_nodes)<2:return {}
        base=int(hashlib.sha256(prompt.encode()).hexdigest()[:12],16)
        acc=defaultdict(float)
        for r in range(n):
            perm=list(source_nodes);random.Random(base+r*7919).shuffle(perm)
            mapping=dict(zip(source_nodes,perm))
            raw=defaultdict(float)
            for y,w1 in self.out.get(x,{}).items():
                yy=mapping.get(y,y)
                for z,w2 in self.out.get(yy,{}).items():
                    if z==x:continue
                    raw[z]+=math.sqrt(max(0.,w1*w2))
            mx=max(raw.values(),default=0.)+1e-12
            for z,v in raw.items():acc[z]+=float(v/mx)/n
        return dict(acc)

class GraphBindingSystem:
    """Same replay bank as v44_multi_auto plus a compact identity graph."""
    def __init__(self,replay_bank,graph):
        self.replay_bank=replay_bank;self.identity_graph=graph
        # bank-like audit surface
        self.capacity=replay_bank.capacity;self.slots=replay_bank.slots
        self.writes=replay_bank.writes;self.replacements=replay_bank.replacements
        self.blocked=replay_bank.blocked;self.holds=replay_bank.holds;self.margin=getattr(replay_bank,'margin',0.)
    def corrupt(self,p,seed):self.replay_bank.corrupt(p,seed)
    def replay_ids(self,parity=True):return self.replay_bank.replay_ids(parity)
    def retrieve_templates(self,prompt,max_hops=3,parity=True):return self.replay_bank.retrieve_templates(prompt,max_hops,parity)
    def identity_join(self,prompt,parity=True,shuffle=False):
        if shuffle:
            sc=self.identity_graph.shuffled_two_step_mean(prompt,32);return sc,[],{'resolved':bool(sc),'control':'32-shuffle-mean'}
        sc,paths=self.identity_graph.two_step(prompt);return sc,paths,{'resolved':bool(sc),'edges':len(self.identity_graph.edges)}

def fused_candidate_rank(model,vocab,prompt,true_answer,candidates,struct_scores=None,struct_gain=1.35):
    v=logits_for_prompt(model,vocab,prompt)
    base=np.array([float(v[vocab.stoi.get(a,1)].item()) for a in candidates],float)
    sd=float(np.std(base)+1e-6)
    ss=np.array([0. if struct_scores is None else struct_scores.get(a,0.) for a in candidates],float)
    fused=base+struct_gain*sd*ss
    order=np.argsort(-fused);ranked=[candidates[i] for i in order]
    return ranked.index(true_answer)+1,{candidates[i]:float(fused[i]) for i in range(len(candidates))}

def v45_probe_panel(model,vocab,chains,seed,variant,bank=None,parity=True):
    rows=[];q=chains[chains.kind=='probe'].reset_index(drop=True);zcands=q.z.tolist()
    def add(kind,prompt,ans,item,mode='plain',candidates=None,wrong=None):
        aug=prompt;retr=[];logs=[];struct={};paths=[];meta={}
        if mode=='text_assist' and bank is not None:
            retr,logs=bank.retrieve_templates(prompt,3,parity);aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        elif mode=='second_only' and bank is not None:
            rr,ll=bank.retrieve_templates(prompt,3,parity);retr=rr[-1:] if rr else [];logs=ll
            aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        elif mode in ('identity_join','identity_shuffle') and bank is not None and hasattr(bank,'identity_join'):
            struct,paths,meta=bank.identity_join(prompt,parity,mode=='identity_shuffle')
        r=token_rank(model,vocab,aug,ans);cr=np.nan;margin=np.nan
        if candidates:
            if mode in ('identity_join','identity_shuffle'):
                cr,sc=fused_candidate_rank(model,vocab,aug,ans,candidates,struct)
            else:
                cr,sc=candidate_rank(model,vocab,aug,ans,candidates)
            ts=sc.get(ans,np.nan);ws=sc.get(wrong,np.nan);margin=ts-ws
        else:ts=ws=np.nan
        rows.append({'seed':seed,'variant':variant,'probe':kind,'mode':mode,'item':item,'answer':ans,
                     'rank':r,'candidate_rank':cr,'binding_margin':margin,
                     'candidate_top1':int(cr==1) if np.isfinite(cr) else np.nan,
                     'retrieved':' || '.join(retr),'join_resolved':bool(meta.get('resolved',False)),
                     'join_paths':json.dumps(paths),'struct_true_score':float(struct.get(ans,0.)) if candidates else np.nan,
                     'struct_wrong_score':float(struct.get(wrong,0.)) if candidates and wrong else np.nan})
    for i,(p,a) in enumerate(FACTS):add('exact_fact',p,a,f'f{i}')
    for i,(p,a) in enumerate(REPHRASE):add('rephrase',p,a,f'r{i}')
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        for mode in ['plain','text_assist','second_only','identity_join','identity_shuffle']:
            add('two_hop',f'{row.x} ultimately maps to',row.z,f'c{i}',mode,zcands,wrong.z)
        add('premise_contains',f'{row.x} contains',row.y,f'c{i}')
        add('premise_maps',f'{row.y} maps to',row.z,f'c{i}')
    return pd.DataFrame(rows)

V45_VARIANTS=['baseline','v43_factor_gate','v44_multi_auto','v45_multi_access','v45_pareto_access','v45_structural_graph']

def run_one_v45(seed=0,full=False,corpus='interleaved',device=None,capacity=96,contextual_demo=False):
    seed_all(seed);train,chains=make_training_corpus(seed,contextual_demo=contextual_demo)
    stream=natural_stream(seed,corpus,700) if full else proxy_stream(seed,180);vocab=Vocab(train+stream)
    if device is None:device='cuda' if full and torch.cuda.is_available() else 'cpu'

    base=TinySSM(len(vocab.itos)).to(device);opt=torch.optim.Adam(base.parameters(),lr=.006);order=list(range(len(train)))
    early_state=mid_state=None
    for ep in range(3):
        random.Random(seed+ep).shuffle(order)
        for i in order:train_sentence(base,opt,vocab.encode(train[i]))
        if ep==0:early_state=copy.deepcopy(base.state_dict())
        if ep==1:mid_state=copy.deepcopy(base.state_dict())
    final_state=copy.deepcopy(base.state_dict())
    early=TinySSM(len(vocab.itos)).to(device);early.load_state_dict(early_state)
    mid=TinySSM(len(vocab.itos)).to(device);mid.load_state_dict(mid_state)
    cand=add_stability(unique_candidate_table(train,vocab,early,mid,base))
    cand_v43,meta_v43=evidence_model(cand,'factor');cand_multi,meta_multi=multi_consequence_model(cand,'factor')
    v45bank,merged=build_multi_access_bank(cand_v43,cand_multi,vocab,capacity,True)
    paretobank,_=build_pareto_multi_access_bank(cand_v43,cand_multi,vocab,capacity)
    graphbank=GraphBindingSystem(build_v44_bank(cand_multi,vocab,'multi_auto',capacity),IdentityBindingGraph(cand_v43,512))
    banks={'baseline':None,'v43_factor_gate':build_bank(cand_v43,meta_v43,vocab,'factor_gate',capacity),
           'v44_multi_auto':build_v44_bank(cand_multi,vocab,'multi_auto',capacity),
           'v45_multi_access':v45bank,'v45_pareto_access':paretobank,'v45_structural_graph':graphbank}

    runs=[];probes=[];audit=[]
    qp=chains[chains.kind=='probe'].reset_index(drop=True);prem=set()
    for _,r in qp.iterrows():prem|={f'{r.x} contains {r.y}.',f'{r.y} maps to {r.z}.'}
    for variant in V45_VARIANTS:
        model=TinySSM(len(vocab.itos)).to(device);model.load_state_dict(final_state);opt=torch.optim.Adam(model.parameters(),lr=.006)
        bank=banks[variant];repair=lost=0;replay=[]
        if bank is not None:
            bank.corrupt(.18,seed+1234);replay,repair,lost=bank.replay_ids(True)
        total=0
        for ep in range(2 if full else 1):
            for s in stream:
                train_sentence(model,opt,vocab.encode(s));total+=1
                if replay and total%6==0:train_sentence(model,opt,replay[(total//6)%len(replay)])
        post=v45_probe_panel(model,vocab,chains,seed,variant,bank,True);probes.append(post)
        def mrr(k,m='plain'):
            x=post[(post.probe==k)&(post['mode']==m)]['rank'].to_numpy(float);return float(np.mean(1/x)) if len(x) else np.nan
        def cmrr(m):
            x=post[(post.probe=='two_hop')&(post['mode']==m)].candidate_rank.to_numpy(float);return float(np.nanmean(1/x))
        def bmar(m):
            return float(np.nanmean(post[(post.probe=='two_hop')&(post['mode']==m)].binding_margin))
        jq=post[(post.probe=='two_hop')&(post['mode']=='identity_join')]
        if bank is not None:
            stored={s.text for s in bank.slots}
            both=np.mean([f'{r.x} contains {r.y}.' in stored and f'{r.y} maps to {r.z}.' in stored for _,r in qp.iterrows()])
            audit.append({'seed':seed,'variant':variant,'slots':len(bank.slots),'writes':bank.writes,'replacements':bank.replacements,
                          'blocked':bank.blocked,'holds':bank.holds,'premise_template_coverage':len(stored&prem)/len(prem),
                          'both_premises_stored':both,'repair':repair,'lost':lost})
        runs.append({'seed':seed,'variant':variant,'exact_mrr':mrr('exact_fact'),'rephrase_mrr':mrr('rephrase'),
                     'premise_contains_mrr':mrr('premise_contains'),'premise_maps_mrr':mrr('premise_maps'),
                     'plain_candidate_mrr':cmrr('plain'),'text_assist_candidate_mrr':cmrr('text_assist'),
                     'second_only_candidate_mrr':cmrr('second_only'),'identity_join_candidate_mrr':cmrr('identity_join'),
                     'identity_shuffle_candidate_mrr':cmrr('identity_shuffle'),
                     'identity_join_binding_margin':bmar('identity_join'),'identity_shuffle_binding_margin':bmar('identity_shuffle'),
                     'identity_gain_over_best_nonjoin':cmrr('identity_join')-max(cmrr('plain'),cmrr('text_assist'),cmrr('second_only')),
                     'identity_control_gap':cmrr('identity_join')-cmrr('identity_shuffle'),
                     'join_resolved_rate':float(jq.join_resolved.mean()),'join_true_struct_support':float(jq.struct_true_score.mean()),
                     'join_wrong_struct_support':float(jq.struct_wrong_score.mean()),'parity_repaired':repair,'parity_lost':lost,
                     'stream_loss':float(np.mean([seq_loss(model,vocab.encode(s)) for s in stream[-60:]]))})
    return pd.DataFrame(runs),pd.concat(probes,ignore_index=True),pd.DataFrame(audit),merged,{'v43':meta_v43,'v44':meta_multi}

def main(argv=None):
    ap=argparse.ArgumentParser();ap.add_argument('--full',action='store_true')
    ap.add_argument('--corpus',choices=['shakespeare','bible','interleaved'],default='interleaved')
    ap.add_argument('--seeds',type=int,default=3);ap.add_argument('--capacity',type=int,default=96)
    ap.add_argument('--contextual-demo',action='store_true');args=ap.parse_args(argv)
    RR=[];PP=[];BB=[];MM=[];EE=[]
    for seed in range(args.seeds):
        print('seed',seed)
        r,p,b,m,e=run_one_v45(seed,args.full,args.corpus,capacity=args.capacity,contextual_demo=args.contextual_demo)
        RR.append(r);PP.append(p);BB.append(b);m['seed']=seed;MM.append(m);EE.append(e)
        print(r[['variant','exact_mrr','identity_join_candidate_mrr','identity_join_binding_margin','identity_gain_over_best_nonjoin']].to_string(index=False))
    runs=pd.concat(RR,ignore_index=True);probes=pd.concat(PP,ignore_index=True);banks=pd.concat(BB,ignore_index=True);merged=pd.concat(MM,ignore_index=True)
    tag=(('full_'+args.corpus) if args.full else 'proxy')+('_contextual' if args.contextual_demo else '_legacy')
    runs.to_csv(OUT/f'{tag}_{args.seeds}seed_runs.csv',index=False);probes.to_csv(OUT/f'{tag}_{args.seeds}seed_probe_details.csv',index=False)
    banks.to_csv(OUT/f'{tag}_{args.seeds}seed_bank_audit.csv',index=False);merged.to_csv(OUT/f'{tag}_{args.seeds}seed_multiaccess_values.csv',index=False)
    (OUT/f'{tag}_{args.seeds}seed_evidence_models.json').write_text(json.dumps(EE,indent=2))
    def vals(v,c):return runs[runs.variant==v].sort_values('seed')[c].to_numpy(float)
    checks={'v45_identity_join_margin_positive_seed_fraction':float(np.mean(vals('v45_multi_access','identity_join_binding_margin')>0)),
            'v45_identity_gain_positive_seed_fraction':float(np.mean(vals('v45_multi_access','identity_gain_over_best_nonjoin')>0)),
            'v45_identity_control_gap_positive_seed_fraction':float(np.mean(vals('v45_multi_access','identity_control_gap')>0)),
            'v45_join_resolved_rate_median':float(np.median(vals('v45_multi_access','join_resolved_rate'))),
            'v45_exact_beats_v43_seed_fraction':float(np.mean(vals('v45_multi_access','exact_mrr')>vals('v43_factor_gate','exact_mrr'))),
            'v45_exact_beats_v44_auto_seed_fraction':float(np.mean(vals('v45_multi_access','exact_mrr')>vals('v44_multi_auto','exact_mrr'))),
            'v45_pareto_identity_join_margin_positive_seed_fraction':float(np.mean(vals('v45_pareto_access','identity_join_binding_margin')>0)),
            'v45_pareto_identity_gain_positive_seed_fraction':float(np.mean(vals('v45_pareto_access','identity_gain_over_best_nonjoin')>0)),
            'v45_pareto_identity_control_gap_positive_seed_fraction':float(np.mean(vals('v45_pareto_access','identity_control_gap')>0)),
            'v45_pareto_join_resolved_rate_median':float(np.median(vals('v45_pareto_access','join_resolved_rate'))),
            'v45_graph_binding_margin_positive_seed_fraction':float(np.mean(vals('v45_structural_graph','identity_join_binding_margin')>0)),
            'v45_graph_identity_gain_positive_seed_fraction':float(np.mean(vals('v45_structural_graph','identity_gain_over_best_nonjoin')>0)),
            'v45_graph_control_gap_positive_seed_fraction':float(np.mean(vals('v45_structural_graph','identity_control_gap')>0)),
            'v45_graph_join_resolved_rate_median':float(np.median(vals('v45_structural_graph','join_resolved_rate'))),
            'parity_repairs_faults':bool(np.median(vals('v45_structural_graph','parity_repaired'))>0)}
    (OUT/f'{tag}_{args.seeds}seed_checks.json').write_text(json.dumps(checks,indent=2))
    med=runs.groupby('variant').median(numeric_only=True);bankmed=banks.groupby('variant').median(numeric_only=True)
    report=f"""Adrianic v4.5 — Multi-Access Memory + Identity Binding State

Mode: {tag}
Seeds: {args.seeds}
Capacity: {args.capacity}
Contextual demonstration ingestion: {args.contextual_demo}

Each memory object has independent consolidation/replay value C and retrieval/address value R.
Replay uses C; query addressing uses R. Resource survival may be earned by either role.

The identity-join operator knows no relation names, semantic roles, chain IDs, target labels or answer keys.
It uses rare query anchors, exact shared-token identity across two templates, observed sequence order and learned retrieval access.

The composed X->Z target sentence remains absent for every probe chain.

Median metrics
==============
{med.to_string()}

Median bank audit
=================
{bankmed.to_string()}

Checks
======
{json.dumps(checks,indent=2)}

Interpretation boundary
=======================
A positive identity-join result is SYSTEM-level deterministic compositional binding.
It is not evidence that the RNN independently induced first-order logic internally.

Critical controls:
- identity join must beat plain/text/second-only;
- identity join should beat identity-shuffled support;
- no probe composed target is present in training;
- replay and retrieval values remain separately logged.
"""
    (OUT/f'{tag}_{args.seeds}seed_REPORT.txt').write_text(report);print(report)



## Full natural-corpus run

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])
```

In [ ]:
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])

## Larger replication after mechanics pass

```python
main(["--full", "--corpus", "interleaved", "--seeds", "8", "--contextual-demo"])
```

## Verified local eight-seed result

# Adrianic v4.5 — Multi-Access Memory + Explicit Identity-Binding State

## Eight-seed local proxy verification

v4.5 was built after v4.4 separated two problems that one memory score had been trying to solve:

- **consolidation/replay value** — what deserves to influence long-term learning;
- **query/retrieval value** — what is useful for resolving the current state or question.

The first v4.5 screen exposed another coupling bug: evidence confidence was being allowed to suppress an already evidence-adjusted access value a second time. That destroyed retrieval coverage. Confidence was therefore returned to its proper job: unresolved/HOLD control and audit, with only a weak access modulation.

A direct multi-access full-template bank improved after that correction, but it still could not reliably preserve both replay-optimal material and every useful relational premise inside the same 96 raw-template slots.

The final local v4.5 architecture therefore adds a **compact structural identity graph** alongside the raw-template bank.

This is not a second full memory bank. It stores recurrence-qualified directed token-identity edges derived from observed sequences.

## Structural binding primitive

For recurring rare tokens, observed order creates directed edges.

For a probe chain trained only on:

`X ... Y`

and

`Y ... Z`

the structural state can contain:

`X -> Y`

and

`Y -> Z`

without ever storing the omitted composed probe target `X -> Z`.

At query time, a deterministic two-step traversal can therefore produce a candidate Z through the **exact shared identity Y**.

The operator receives no semantic relation names, chain IDs, fact labels, answer keys or target labels.

## Critical negative control

A strong bridge-identity shuffle preserves:

- the first-hop edges,
- the collection of second-hop adjacency lists,
- the edge-weight distribution,

but permutes which second-hop adjacency list is attached to each intermediate identity.

This destroys exact bridge identity while preserving the gross graph statistics.

## Eight-seed median metrics

```text
                      exact_mrr  rephrase_mrr  plain_candidate_mrr  text_assist_candidate_mrr  second_only_candidate_mrr  identity_join_candidate_mrr  identity_shuffle_candidate_mrr  identity_join_binding_margin  identity_shuffle_binding_margin  identity_gain_over_best_nonjoin  identity_control_gap  join_resolved_rate  stream_loss  parity_repaired  parity_lost
variant                                                                                                                                                                                                                                                                                                                                                                   
baseline               0.387067      0.047078             0.304425                   0.304425                   0.304425                     0.304425                        0.304425                     -0.068553                        -0.068553                         0.000000              0.000000                0.00     0.883345              0.0          0.0
v43_factor_gate        0.369026      0.037470             0.267758                   0.266389                   0.284762                     0.267758                        0.267758                      0.164848                         0.164848                        -0.043333              0.000000                0.00     0.878367             18.5          0.0
v44_multi_auto         0.437326      0.084960             0.350456                   0.350456                   0.350456                     0.350456                        0.350456                      0.083197                         0.083197                         0.000000              0.000000                0.00     0.876543             17.5          0.0
v45_multi_access       0.415312      0.037164             0.309821                   0.329484                   0.295179                     0.315536                        0.347480                      0.309622                         0.365705                        -0.008909             -0.031667                0.65     0.875978             18.0          0.0
v45_pareto_access      0.468716      0.038601             0.318591                   0.319504                   0.319504                     0.320258                        0.322758                      0.248128                         0.242468                        -0.012381             -0.001587                0.40     0.875953             17.5          0.0
v45_structural_graph   0.437326      0.084960             0.350456                   0.350456                   0.350456                     0.594643                        0.351488                      1.292058                         0.102728                         0.262044              0.263234                1.00     0.876543             17.5          0.0
```

## Headline structural-graph result

For `v45_structural_graph`:

- identity-join binding margin was positive in **100%** of seeds;
- identity join beat the best non-join control in **100%** of seeds;
- identity join beat the bridge-shuffled control in **100%** of seeds;
- median join-resolution rate was **100%**;
- identity-join candidate MRR beat the same replay learner without structural composition in **100%** of seeds.

Median candidate MRR:

- v44 multi-consequence learner, no structural join: **0.350**
- v45 structural identity join: **0.595**
- bridge-shuffled structural control: **0.351**

Median true-vs-matched-wrong margin:

- identity join: **1.292**
- bridge-shuffled control: **0.103**

Median strict gain over the best of plain, text-assist and second-only controls:
**0.262**

The v45 structural variant uses the **same v44 multi-auto replay bank**. Exact-retention results matched to a maximum absolute difference of
**0** across eight seeds, so the composition gain is not coming from a different replay curriculum.

## What did NOT work as well

The raw-template `v45_multi_access` and Pareto variants improved retrieval coverage relative to the first screen, but they were not robust enough to replace the compact structural representation.

That result matters: merely attaching two scalar access values to the same limited raw-template bank is not sufficient.

The useful structural information is cheaper to preserve in a derived relation/identity state than as duplicate full episodic templates.

## Interpretation boundary

This is the first v4.x local result for which the **system-level composition criterion** is meaningfully different from copy/retrieval:

1. the probe X->Z sentence is absent from training;
2. the two-step identity join beats second-only / text-assist behavior;
3. exact bridge identity beats a structure-preserving identity shuffle;
4. the structural variant shares the same replay learner as its no-join control.

This is evidence for a deterministic **system-level compositional binding mechanism** in the local controlled test.

It is **not** evidence that the RNN itself spontaneously learned first-order logic, and it is not yet evidence that the effect survives natural Shakespeare/KJV interference.

## Natural-corpus next test

Run:

```python
main(["--full", "--corpus", "interleaved", "--seeds", "3", "--contextual-demo"])
```

The full run is the next gate. If the identity-control gap and strict composition gain survive there, the result deserves larger-seed replication and relation-grammar permutation tests.
